In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

In [2]:
df = pd.read_csv('../datasets/datasets_smiles_approval.csv')

In [3]:
df

,Name,SMILES,DRUGBANK_ID,ApprovalDate,THROMBOCYTOPENIA
0,cefaclor,[H][C@]12SCC(Cl)=C(N1C(=O)[C@H]2NC(=O)[C@H](N)...,DB00833,1979-07-28,1
1,hydrochlorothiazide,NS(=O)(=O)C1=C(Cl)C=C2NCNS(=O)(=O)C2=C1,DB00999,1981-08-20,1
2,dicloxacillin,[H][C@]12SC(C)(C)[C@@H](N1C(=O)[C@H]2NC(=O)C1=...,DB00485,1982-06-03,1
3,trimethoprim,COC1=CC(CC2=CN=C(N)N=C2N)=CC(OC)=C1OC,DB00440,1983-05-09,0
4,ceftazidime,[O-]C(=O)C1=C(CS[C@]2([H])[C@H](NC(=O)C(=N/OC(...,DB00438,1985-11-20,1
...,...,...,...,...,...
293,pivampicillin,[H][C@]12SC(C)(C)[C@@H](N1C(=O)[C@H]2NC(=O)[C@...,DB01604,NaN,1
294,pivmecillinam,[H]C(=N[C@@H]1C(=O)N2[C@@H](C(=O)OCOC(=O)C(C)(...,DB01605,NaN,1
295,quinidine,[H][C@@]12CCN(C[C@@H]1C=C)[C@]([H])(C2)[C@@H](...,DB00908,NaN,1
296,teicoplanin,CCCCCCCCCC(=O)N[C@@H]1[C@@H](O)[C@H](O)[C@@H](...,DB06149,NaN,1


In [4]:
data = []

for drug_id in tqdm(df['DRUGBANK_ID']):
    url = f"https://go.drugbank.com/targets?approved=0&nutraceutical=0&illicit=0&investigational=0&withdrawn=0&experimental=0&us=0&ca=0&eu=0&q%5Bdrug%5D={drug_id}&q%5Bassociation_type%5D=target&q%5Bpolypeptides.name%5D=&button="
    
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    table_rows = soup.select("#targets-table > tbody > tr")
    
    for row in table_rows:
        target_name = row.select_one("td:nth-child(3) > a").text.strip()
        data.append([drug_id, target_name])

100%|██████████| 298/298 [01:46<00:00,  2.79it/s]


In [5]:
df_target = pd.DataFrame(data, columns=['drug_id', 'target_nm'])

In [6]:
df_target

,drug_id,target_nm
0,DB00833,Penicillin-binding protein 3
1,DB00833,Penicillin-binding protein 1A
2,DB00999,Calcium-activated potassium channel subunit al...
3,DB00999,Solute carrier family 12 member 3
4,DB00485,Penicillin-binding protein 3
...,...,...
1011,DB00908,Alpha-1D adrenergic receptor
1012,DB00908,Potassium voltage-gated channel subfamily H me...
1013,DB00908,Alpha-1B adrenergic receptor
1014,DB06149,D-Ala-D-Ala moiety of NAM/NAG peptide subunits...


In [13]:
df_target.to_csv('../datasets/target_data_0722.csv', index = False)

In [35]:
crosstab = pd.crosstab(df_target['drug_id'], df_target['target_nm'])

In [37]:
crosstab_reset = crosstab.reset_index()

In [41]:
crosstab_reset.index = range(len(crosstab_reset))

In [45]:
crosstab_reset

target_nm,drug_id,"1,3-beta-glucan synthase component FKS1",16S ribosomal RNA,"2-oxoglutarate dehydrogenase, mitochondrial",23S ribosomal RNA,"3-beta-hydroxysteroid-Delta(8),Delta(7)-isomerase",3-hydroxy-3-methylglutaryl-coenzyme A reductase,3-phosphoinositide-dependent protein kinase 1,30S ribosomal protein,30S ribosomal protein S10,...,Voltage-gated Potassium Channels,Voltage-gated sodium channel alpha subunit,alpha1-acid glycoprotein,"cAMP and cAMP-inhibited cGMP 3',5'-cyclic phosphodiesterase 10A","cAMP-specific 3',5'-cyclic phosphodiesterase 4A","cAMP-specific 3',5'-cyclic phosphodiesterase 4B","cAMP-specific 3',5'-cyclic phosphodiesterase 4C","cAMP-specific 3',5'-cyclic phosphodiesterase 4D","cGMP-inhibited 3',5'-cyclic phosphodiesterase A","cGMP-specific 3',5'-cyclic phosphodiesterase"
0,DB00080,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,DB00175,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,DB00188,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,DB00196,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,DB00198,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,DB09101,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
294,DB09319,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
295,DB11367,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
296,DB11641,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [47]:
crosstab_reset.to_csv('../datasets/target_pivot.csv',index=False)

In [54]:
df_merge = pd.merge(df,crosstab_reset, how='left', left_on='DRUGBANK_ID', right_on='drug_id')

In [62]:
df_merge

,Name,SMILES,DRUGBANK_ID,ApprovalDate,THROMBOCYTOPENIA,drug_id,"1,3-beta-glucan synthase component FKS1",16S ribosomal RNA,"2-oxoglutarate dehydrogenase, mitochondrial",23S ribosomal RNA,...,Voltage-gated Potassium Channels,Voltage-gated sodium channel alpha subunit,alpha1-acid glycoprotein,"cAMP and cAMP-inhibited cGMP 3',5'-cyclic phosphodiesterase 10A","cAMP-specific 3',5'-cyclic phosphodiesterase 4A","cAMP-specific 3',5'-cyclic phosphodiesterase 4B","cAMP-specific 3',5'-cyclic phosphodiesterase 4C","cAMP-specific 3',5'-cyclic phosphodiesterase 4D","cGMP-inhibited 3',5'-cyclic phosphodiesterase A","cGMP-specific 3',5'-cyclic phosphodiesterase"
0,cefaclor,[H][C@]12SCC(Cl)=C(N1C(=O)[C@H]2NC(=O)[C@H](N)...,DB00833,1979-07-28,1,DB00833,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,hydrochlorothiazide,NS(=O)(=O)C1=C(Cl)C=C2NCNS(=O)(=O)C2=C1,DB00999,1981-08-20,1,DB00999,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,dicloxacillin,[H][C@]12SC(C)(C)[C@@H](N1C(=O)[C@H]2NC(=O)C1=...,DB00485,1982-06-03,1,DB00485,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,trimethoprim,COC1=CC(CC2=CN=C(N)N=C2N)=CC(OC)=C1OC,DB00440,1983-05-09,0,DB00440,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ceftazidime,[O-]C(=O)C1=C(CS[C@]2([H])[C@H](NC(=O)C(=N/OC(...,DB00438,1985-11-20,1,DB00438,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,pivampicillin,[H][C@]12SC(C)(C)[C@@H](N1C(=O)[C@H]2NC(=O)[C@...,DB01604,NaN,1,DB01604,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
294,pivmecillinam,[H]C(=N[C@@H]1C(=O)N2[C@@H](C(=O)OCOC(=O)C(C)(...,DB01605,NaN,1,DB01605,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
295,quinidine,[H][C@@]12CCN(C[C@@H]1C=C)[C@]([H])(C2)[C@@H](...,DB00908,NaN,1,DB00908,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
296,teicoplanin,CCCCCCCCCC(=O)N[C@@H]1[C@@H](O)[C@H](O)[C@@H](...,DB06149,NaN,1,DB06149,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [68]:
df_merge.drop(['drug_id'], axis = 1, inplace=True)

In [70]:
df_merge

,Name,SMILES,DRUGBANK_ID,ApprovalDate,THROMBOCYTOPENIA,"1,3-beta-glucan synthase component FKS1",16S ribosomal RNA,"2-oxoglutarate dehydrogenase, mitochondrial",23S ribosomal RNA,"3-beta-hydroxysteroid-Delta(8),Delta(7)-isomerase",...,Voltage-gated Potassium Channels,Voltage-gated sodium channel alpha subunit,alpha1-acid glycoprotein,"cAMP and cAMP-inhibited cGMP 3',5'-cyclic phosphodiesterase 10A","cAMP-specific 3',5'-cyclic phosphodiesterase 4A","cAMP-specific 3',5'-cyclic phosphodiesterase 4B","cAMP-specific 3',5'-cyclic phosphodiesterase 4C","cAMP-specific 3',5'-cyclic phosphodiesterase 4D","cGMP-inhibited 3',5'-cyclic phosphodiesterase A","cGMP-specific 3',5'-cyclic phosphodiesterase"
0,cefaclor,[H][C@]12SCC(Cl)=C(N1C(=O)[C@H]2NC(=O)[C@H](N)...,DB00833,1979-07-28,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,hydrochlorothiazide,NS(=O)(=O)C1=C(Cl)C=C2NCNS(=O)(=O)C2=C1,DB00999,1981-08-20,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,dicloxacillin,[H][C@]12SC(C)(C)[C@@H](N1C(=O)[C@H]2NC(=O)C1=...,DB00485,1982-06-03,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,trimethoprim,COC1=CC(CC2=CN=C(N)N=C2N)=CC(OC)=C1OC,DB00440,1983-05-09,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ceftazidime,[O-]C(=O)C1=C(CS[C@]2([H])[C@H](NC(=O)C(=N/OC(...,DB00438,1985-11-20,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,pivampicillin,[H][C@]12SC(C)(C)[C@@H](N1C(=O)[C@H]2NC(=O)[C@...,DB01604,NaN,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
294,pivmecillinam,[H]C(=N[C@@H]1C(=O)N2[C@@H](C(=O)OCOC(=O)C(C)(...,DB01605,NaN,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
295,quinidine,[H][C@@]12CCN(C[C@@H]1C=C)[C@]([H])(C2)[C@@H](...,DB00908,NaN,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
296,teicoplanin,CCCCCCCCCC(=O)N[C@@H]1[C@@H](O)[C@H](O)[C@@H](...,DB06149,NaN,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [72]:
df_merge.to_csv('../datasets/dataset_0722.csv',index = False)